# VoiceShield AI

AASIST speech detection with a controlled bandwidth-adaptation experiment.

## 1. Test the saved model

Run the four steps in this section, in order. They load the saved bandwidth-adapted model and test two development recordings. No training is needed

**Colab:** choose a T4 runtime and upload VoiceShield.zip and LA_zenodo.zip through the Files panel.

**Local VS Code:** open the folder containing both ZIP files as your workspace, select your Python environment and install requirements.txt. Set VOICESHIELD_ROOT to the ZIP folder if your kernel starts elsewhere. Git must be available. The setup prints the selected device and folder.


### 1.1 Set up the environment
Choose the file location, download the pinned AASIST source and select CPU or GPU. Run once per session.


In [6]:
from pathlib import Path
import os, sys, subprocess, json, csv, hashlib, shutil

# Use the same project layout in Colab and on the laptop.
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

BASE = (
    Path('/content') if IN_COLAB
    else Path(os.environ.get('VOICESHIELD_ROOT', Path.cwd()))
)
BASE = BASE.resolve()
BASE.mkdir(parents=True, exist_ok=True)
PROJECT = BASE / 'aasist'
ROOT = BASE / 'VoiceShield'
DATA = BASE / 'LA'
print('Environment:', 'Colab' if IN_COLAB else 'Local laptop')
print('Place the project and dataset ZIP files here:', BASE)

git = shutil.which('git')
assert git, 'Git was not found. Install Git and restart VS Code.'
COMMIT = 'a04c9863f63d44471dde8a6abcb3b082b07cd1d1'
if not PROJECT.exists():
    subprocess.run([git, 'clone', 'https://github.com/clovaai/aasist.git', str(PROJECT)], check=True)
if not (PROJECT / '.git').exists():
    raise RuntimeError(f'{PROJECT} is not an AASIST Git checkout. Choose another project folder.')
subprocess.run([git, '-C', str(PROJECT), 'checkout', COMMIT], check=True)

if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

# Install requirements.txt in your local environment before running this cell.
# Colab already provides torch and numpy.
if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'soundfile', 'scipy'], check=True)
import torch
import numpy as np
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Python:', sys.version)
print('PyTorch:', torch.__version__, 'NumPy:', np.__version__)
print('Running on:', device)
if device.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('CPU mode: the two-recording test works more slowly.')


Environment: Colab
Place the project and dataset ZIP files here: /content
Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
PyTorch: 2.14.1+cu130 NumPy: 2.4.6
Running on: cuda
GPU: Tesla T4


### 1.2 Locate the files
Reuse extracted folders where possible otherwise extract the two ZIPs under BASE. The dataset checksum is checked before a new extraction.


In [7]:
import zipfile

FINAL_NAME = 'final_evaluation_20261115T135810961090Z'

# Locate the expected content, including ZIPs with an extra enclosing folder.
def find_folder(search_root, marker):
    marker = Path(marker)
    if (search_root / marker).is_file():
        return search_root
    if not search_root.is_dir():
        return None
    matches = [p for p in search_root.rglob(marker.name)
               if p.as_posix().endswith('/' + marker.as_posix())]
    assert len(matches) <= 1, f'Multiple copies found in {search_root}; choose one folder.'
    return matches[0].parents[len(marker.parts) - 1] if matches else None

def prepare_zip(zip_name, folder_name, marker, expected_md5=None):
    existing = find_folder(BASE / folder_name, marker)
    if existing:
        print('Already extracted:', existing)
        return existing
    unpack_folder = BASE / ('unpacked_' + folder_name)
    existing = find_folder(unpack_folder, marker)
    if existing:
        print('Already extracted:', existing)
        return existing
    archive = BASE / zip_name

    if IN_COLAB and not archive.is_file():
        drive_folder = (
            "https://drive.google.com/drive/folders/"
            "1hwie90TA5Fg8U6blHkxccMpOSL9FdNFX"
        )

        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "--upgrade", "gdown"],
            check=True,
        )
        gdown = [sys.executable, "-m", "gdown"]

        listing = subprocess.check_output(
            [*gdown, drive_folder, "--json", "--quiet"], text=True
        )
        matches = [
            item["url"]
            for item in json.loads(listing)
            if Path(item["path"]).name == zip_name
        ]
        if len(matches) != 1:
            raise RuntimeError(f"Expected one shared file named {zip_name} in Drive.")

        partial = archive.with_suffix(".zip.part")
        print(f"Downloading {zip_name} from Google Drive…", flush=True)
        subprocess.run(
            [*gdown, matches[0], "--continue", "-O", str(partial)],
            check=True,
        )
        if not zipfile.is_zipfile(partial):
            raise RuntimeError(f"Drive did not return a valid ZIP for {zip_name}.")
        partial.replace(archive)

    assert archive.is_file(), f'Place {zip_name} here: {BASE}'
    if expected_md5:
        print('Checking dataset ZIP...')
        digest = hashlib.md5()
        with archive.open('rb') as stream:
            for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b''):
                digest.update(chunk)
        assert digest.hexdigest() == expected_md5, 'Dataset checksum mismatch'
    unpack_folder.mkdir(parents=True, exist_ok=True)
    unpack_folder = unpack_folder.resolve()
    print('Extracting', zip_name, 'to', unpack_folder)
    with zipfile.ZipFile(archive) as z:
        for name in z.namelist():
            assert (unpack_folder / name).resolve().is_relative_to(unpack_folder), 'Unsafe ZIP path'
        z.extractall(unpack_folder)
    found = find_folder(unpack_folder, marker)
    assert found, f'Expected content not found. Check the files inside {archive}'
    return found

ROOT = prepare_zip('VoiceShield.zip', 'VoiceShield', f'{FINAL_NAME}/frozen_plan.json')
DATA = prepare_zip('LA_zenodo.zip', 'LA',
    'ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.dev.trl.txt',
    '30c98f11d8b2bc21f2c257bfd78bb5c5')
assert (ROOT / FINAL_NAME / 'final_metrics.csv').is_file(), 'Final metrics missing from project ZIP'
assert (DATA / 'ASVspoof2019_LA_dev/flac').is_dir(), 'Development audio folder missing'
print('Model/results folder:', ROOT)
print('Dataset folder:', DATA)
print('Files ready. Continue to step 3.')


Already extracted: /content/unpacked_VoiceShield/VoiceShield
Already extracted: /content/unpacked_LA/LA
Model/results folder: /content/unpacked_VoiceShield/VoiceShield
Dataset folder: /content/unpacked_LA/LA
Files ready. Continue to step 3.


### 1.3 Load the bandwidth-adapted model
The default MODEL_NAME is augmented. The saved checkpoint hash is checked before loading. Other saved versions can be selected with original or clean_only


In [8]:
from models.AASIST import Model

MODEL_NAME = 'augmented'  # choices: original, clean_only, augmented
final_folder = ROOT / 'final_evaluation_20261115T135810961090Z'
frozen_plan = json.loads((final_folder / 'frozen_plan.json').read_text())
entry = next(item for item in frozen_plan['models'] if item['name'] == MODEL_NAME)
if MODEL_NAME == 'original':
    model_path = PROJECT / 'models/weights/AASIST.pth'
else:
    # Find the checkpoint inside the uploaded project folder.
    recorded = Path(entry['checkpoint'])
    model_path = ROOT / recorded.parent.name / recorded.name
assert model_path.is_file(), f'Missing submitted checkpoint: {model_path}'
assert hashlib.sha256(model_path.read_bytes()).hexdigest() == entry['checkpoint_sha256'], 'Checkpoint hash mismatch'
model = Model(entry['model_config']).to(device)
print(model.load_state_dict(torch.load(model_path, map_location=device, weights_only=True), strict=True))
model.eval()
print('Loaded:', MODEL_NAME)
print('Clean-development decision threshold:', entry['threshold'])


<All keys matched successfully>
Loaded: augmented
Clean-development decision threshold: -0.48679405450820923


### 1.4 Run a small inference check
One genuine and one spoof development recording are used here. Genuine has label 1; spoof has label 0. A higher bona-fide score means more evidence for genuine speech. Scores are not probabilities and these two examples are not an accuracy estimate


In [9]:
import soundfile as sf
from data_utils import genSpoof_list, Dataset_ASVspoof2019_devNeval

protocol = DATA / 'ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.dev.trl.txt'
labels, ids = genSpoof_list(str(protocol), is_train=False, is_eval=False)
selected = [next(uid for uid in ids if labels[uid] == label) for label in [1, 0]]
audio_folder = DATA / 'ASVspoof2019_LA_dev'
for uid in selected:
    y, sr = sf.read(audio_folder / 'flac' / f'{uid}.flac')
    assert sr == 16000 and y.ndim == 1 and y.size > 0
    assert np.isfinite(y).all()

dataset = Dataset_ASVspoof2019_devNeval(selected, audio_folder)
for index in range(len(dataset)):
    waveform, uid = dataset[index]
    x = waveform.unsqueeze(0).to(device)
    assert tuple(x.shape) == (1, 64600)
    with torch.inference_mode():
        embedding, logits = model(x)
        _, repeated_logits = model(x)
    assert tuple(logits.shape) == (1, 2) and torch.isfinite(logits).all()
    torch.testing.assert_close(logits, repeated_logits)
    print(uid, '| true label:', 'genuine' if labels[uid] == 1 else 'spoof',
          '| bona-fide score:', round(logits[0, 1].item(), 4))


LA_D_1047731 | true label: genuine | bona-fide score: 6.1072
LA_D_1008730 | true label: spoof | bona-fide score: -2.4711


The model test ends here. The next section documents the experiment and reads its saved evidence. It does not require more training.

## 2. Experiment record and evidence

The main results come from the completed Colab experiment. A later laptop repeat completed training and development scoring but was stopped during final evaluation. It is kept separately under runs and is not mixed into the main results

### 2.1 Saved final results

The following cell reads the completed six row results file. It does not rerun evaluation. EER and error rates are saved as fractions and displayed as percentages. Each model uses its own clean development decision threshold


In [10]:
metrics_file = final_folder / 'final_metrics.csv'
with metrics_file.open(newline='') as stream:
    saved_metrics = list(csv.DictReader(stream))
assert len(saved_metrics) == 6
for row in saved_metrics:
    print(f"{row['model']:12s} {row['condition']:12s} | EER {float(row['eer'])*100:.3f}% | genuine flagged {row['genuine_flagged']}/{row['genuine_total']} | spoof missed {row['spoof_missed']}/{row['spoof_total']}")
print('Quick demonstration complete. No training or evaluation files were changed.')


original     clean        | EER 0.830% | genuine flagged 12/7355 | spoof missed 6339/63882
original     bandlimited  | EER 4.051% | genuine flagged 255/7355 | spoof missed 2919/63882
clean_only   clean        | EER 0.882% | genuine flagged 13/7355 | spoof missed 4917/63882
clean_only   bandlimited  | EER 4.025% | genuine flagged 274/7355 | spoof missed 2757/63882
augmented    clean        | EER 1.047% | genuine flagged 14/7355 | spoof missed 6413/63882
augmented    bandlimited  | EER 3.900% | genuine flagged 162/7355 | spoof missed 3582/63882
Quick demonstration complete. No training or evaluation files were changed.


### 2.2 Saved training logs
Read the two original training logs, settings and selected IDs from the project ZIP. This also checks that both training runs used the same 4,000 ID


In [11]:
for name, folder_name in [
    ('Bandwidth-augmented', 'adaptation_trial_20261115T131001473413Z'),
    ('Clean-only control', 'clean_control_20261115T133443770992Z')
]:
    trial = ROOT / folder_name
    settings = json.loads((trial / 'settings.json').read_text())
    log = json.loads((trial / 'training_log.json').read_text())
    training_ids = (trial / 'training_ids.txt').read_text().splitlines()
    assert len(training_ids) == len(set(training_ids)) == 4000
    print('\n' + name)
    print('Learning rate:', settings['learning_rate'])
    print('Batch size:', settings['batch_size'], '| epochs:', settings['epochs'])
    print('Augmentation probability:', settings['augmentation_probability'])
    print('Training log:', log)
    print('Training ID saved:', len(training_ids))

assert (ROOT / 'adaptation_trial_20261115T131001473413Z/training_ids.txt').read_text() == (
    ROOT / 'clean_control_20261115T133443770992Z/training_ids.txt').read_text(), 'Training subsets differ'
print('PASS: both runs used the same saved training IDs.')
print('Pinned AASIST source commit:', frozen_plan['source_commit'])



Bandwidth-augmented
Learning rate: 1e-06
Batch size: 8 | epochs: 1
Augmentation probability: 0.5
Training log: {'epoch': 1, 'recordings': 4000, 'filtered_recordings': 2033, 'batch_loss_mean_weighted_by_batch_size': 0.08916769217137471, 'training_seconds': 309.4998931759999}
Training ID saved: 4000

Clean-only control
Learning rate: 1e-06
Batch size: 8 | epochs: 1
Augmentation probability: 0.0
Training log: {'epoch': 1, 'recordings': 4000, 'filtered_recordings': 0, 'batch_loss_mean_weighted_by_batch_size': 0.022070070423701623, 'training_seconds': 310.01259834699977}
Training ID saved: 4000
PASS: both runs used the same saved training IDs.
Pinned AASIST source commit: a04c9863f63d44471dde8a6abcb3b082b07cd1d1


### 2.3 Method and reported outputs

#### What changed in the audio

Both genuine and spoof recordings were processed with the same 300-3400 Hz band-pass filter. The implementation used scipy.signal.butter(N=6, ..., output='sos') and sosfilt before cropping. Audio stayed mono at 16 kHz

AASIST takes the first 64,600 samples for scoring, repeating shorter audio. During fine-tuning, longer audio was randomly cropped. Labels: spoof=0, genuine=1.

#### Experiment sequence

1. Load the official pretrained AASIST checkpoint and verify inference.
2. Check a reproducible 500-recording development subset.
3. Score the complete development partition: 24,844 recordings, clean and filtered.
4. Save a training subset of 4,000 IDs: 407 genuine and 3,593 spoof, seed 42.
5. Fine-tune a fresh checkpoint copy for one epoch with 50% filtering probability.
6. Run a matched clean-only fine-tuning control from the same original checkpoint.
7. Compare development performance and freeze checkpoints/settings/thresholds.
8. Score all three models on 71,237 held-out evaluation recordings in both conditions.

Fine-tuning used Adam, learning rate 1e-6, batch size 8 and weighted cross-entropy (spoof weight 0.1, genuine weight 0.9). Normal training updated BatchNorm statistics. Each model's clean development scores selected its decision threshold. Evaluation EER swept thresholds for measurement only, its threshold was discarded.

#### Recorded outputs

The entries below summarise the completed Colab run. Section 2.4 recomputes the evaluation tables from the saved scores, and sections 2.1 and 2.2 read the archived metrics, settings and training logs from VoiceShield.zip.

| Training run | Epochs | Recordings | Filtered | Seconds | Logged average batch loss |
|---|---:|---:|---:|---:|---:|
| Augmented | 1 | 4000 | 2033 | 309.500 | 0.089168 |
| Clean-only control | 1 | 4000 | 0 | 310.013 | 0.022070 |

| Model | Development clean EER (%) | Development filtered EER (%) | Final clean EER (%) | Final filtered EER (%) |
|---|---:|---:|---:|---:|
| Original | 0.511 | 1.680 | 0.830 | 4.051 |
| Clean-only | 0.431 | 1.806 | 0.882 | 4.025 |
| Augmented | 0.546 | 1.375 | 1.047 | 3.900 |

| Model / evaluation condition | Genuine flagged / 7355 | Spoof missed / 63882 |
|---|---:|---:|
| Original / clean | 12 | 6339 |
| Original / filtered | 255 | 2919 |
| Clean-only / clean | 13 | 4917 |
| Clean-only / filtered | 274 | 2757 |
| Augmented / clean | 14 | 6413 |
| Augmented / filtered | 162 | 3582 |

These counts use each model's own development selected threshold. They are not error counts at the evaluation EER threshold. One seed and one bandwidth condition do not establish statistical significance, real-call reliability or general modern voice-cloning detection. These limits apply when interpreting the results

VoiceShield.zip contains training IDs, settings, logs, checkpoint hashes, per-recording scores and frozen_plan.json.

#### Optional: standalone baseline script

Section 2.4 already recomputes the baselines inside this notebook. The separate VoiceShield_Baseline_Checks.zip also contains the existing reproduction script, its README and the recorded baseline outputs. Extract it alongside VoiceShield.zip, then run the command below from that folder. The LA and aasist arguments may be absolute paths. Use the dataset path printed in section 1.2. This check reads saved labels and does not load audio or train a model.

```text
python check_trivial_baselines.py --archive VoiceShield.zip --data unpacked_LA/LA --aasist aasist --output baseline_reproduction
```


### 2.4 Independent check from the saved scores (about 2 minutes, CPU only)

This section needs steps 1.1 to 1.3 to have been run. It does not load a model or any audio. It uses only the per-recording score files in the project ZIP, so it also works on a CPU runtime.

1. **Recompute** the final EERs and the fixed-threshold error counts from the 71,237 saved evaluation scores and check them against `final_metrics.csv`.
2. **Bootstrap** 95% confidence intervals by resampling evaluation recordings, with the same resample applied to every model so differences are paired.
3. **Trivial baselines**: majority class and a seeded random scorer, saved to `baseline_sanity_check_20261118`.

The bootstrap only measures uncertainty from which evaluation recordings were drawn. It does not capture variation between training seeds, and recordings from the same speaker or attack are not fully independent.

In [12]:
import numpy as np, pandas as pd, csv

def eer_fast(genuine, spoof):
    # Same algorithm as the official compute_det_curve / compute_eer, kept vectorised for resampling.
    scores = np.concatenate((genuine, spoof))
    labels = np.concatenate((np.ones(genuine.size), np.zeros(spoof.size)))
    labels = labels[np.argsort(scores, kind='mergesort')]
    n = scores.size
    tar = np.cumsum(labels)
    frr = np.concatenate(([0.0], tar / genuine.size))
    far = np.concatenate(([1.0], (spoof.size - (np.arange(1, n + 1) - tar)) / spoof.size))
    i = np.argmin(np.abs(frr - far))
    return (frr[i] + far[i]) / 2

FILES = {('original', 'clean'): 'original_clean_scores.csv',
         ('original', 'filtered'): 'original_bandlimited_scores.csv',
         ('clean_only', 'clean'): 'clean_only_clean_scores.csv',
         ('clean_only', 'filtered'): 'clean_only_bandlimited_scores.csv',
         ('augmented', 'clean'): 'augmented_clean_scores.csv',
         ('augmented', 'filtered'): 'augmented_bandlimited_scores.csv'}
SAVED = {(r['model'], 'filtered' if r['condition'] == 'bandlimited' else 'clean'): r
         for r in csv.DictReader((final_folder / 'final_metrics.csv').open(newline=''))}

tables = {k: pd.read_csv(final_folder / f) for k, f in FILES.items()}
reference = tables[('original', 'clean')]
for k, t in tables.items():   # identical recordings, order and labels everywhere
    assert (t.recording_id == reference.recording_id).all() and (t.true_label == reference.true_label).all(), k
eval_ids = reference.recording_id.values
y = reference.true_label.values
genuine_idx, spoof_idx = np.where(y == 1)[0], np.where(y == 0)[0]
S = {k: t.bona_fide_score.values for k, t in tables.items()}
print(f'Evaluation recordings: {y.size} (genuine {genuine_idx.size}, spoof {spoof_idx.size})\n')

print(f"{'model':11s} {'condition':9s} {'EER %':>7s} {'gen flagged':>11s} {'spoof missed':>12s}   matches saved CSV")
for (model_name, condition), s in S.items():
    threshold = next(m['threshold'] for m in frozen_plan['models'] if m['name'] == model_name)
    eer = eer_fast(s[genuine_idx], s[spoof_idx])
    flagged = int((s[genuine_idx] <= threshold).sum())
    missed = int((s[spoof_idx] > threshold).sum())
    saved = SAVED[(model_name, condition)]
    ok = (abs(eer - float(saved['eer'])) < 1e-12 and flagged == int(saved['genuine_flagged'])
          and missed == int(saved['spoof_missed']))
    assert ok, f'Mismatch for {model_name}/{condition}'
    print(f'{model_name:11s} {condition:9s} {eer*100:7.3f} {flagged:11d} {missed:12d}   PASS')
print('\nPASS: Tables 2 and 3 are reproduced exactly from the saved per-recording scores.')

Evaluation recordings: 71237 (genuine 7355, spoof 63882)

model       condition   EER % gen flagged spoof missed   matches saved CSV
original    clean       0.830          12         6339   PASS
original    filtered    4.051         255         2919   PASS
clean_only  clean       0.882          13         4917   PASS
clean_only  filtered    4.025         274         2757   PASS
augmented   clean       1.047          14         6413   PASS
augmented   filtered    3.900         162         3582   PASS

PASS: Tables 2 and 3 are reproduced exactly from the saved per-recording scores.


In [13]:
# Paired bootstrap over evaluation recordings (genuine and spoof resampled separately).
B, rng = 1000, np.random.default_rng(0)
boot = {k: np.empty(B) for k in S}
for b in range(B):
    g = rng.choice(genuine_idx, genuine_idx.size)
    s_ = rng.choice(spoof_idx, spoof_idx.size)
    for k in S:                      # the same resample is used for every model
        boot[k][b] = eer_fast(S[k][g], S[k][s_])

rows = []
print('95% bootstrap intervals for EER (%)')
for k in S:
    lo, hi = np.percentile(boot[k] * 100, [2.5, 97.5])
    print(f'  {k[0]:11s} {k[1]:9s} {eer_fast(S[k][genuine_idx], S[k][spoof_idx])*100:6.3f}  [{lo:.3f}, {hi:.3f}]')
    rows.append({'comparison': f'{k[0]} {k[1]}', 'estimate_pp': eer_fast(S[k][genuine_idx], S[k][spoof_idx]) * 100,
                 'ci_low_pp': lo, 'ci_high_pp': hi})

print('\nPaired EER differences in percentage points (negative means the first model is better)')
for a, b_ in [(('augmented', 'filtered'), ('clean_only', 'filtered')),
              (('augmented', 'filtered'), ('original', 'filtered')),
              (('clean_only', 'filtered'), ('original', 'filtered')),
              (('augmented', 'clean'), ('clean_only', 'clean')),
              (('augmented', 'clean'), ('original', 'clean'))]:
    point = (eer_fast(S[a][genuine_idx], S[a][spoof_idx]) - eer_fast(S[b_][genuine_idx], S[b_][spoof_idx])) * 100
    d = (boot[a] - boot[b_]) * 100
    lo, hi = np.percentile(d, [2.5, 97.5])
    print(f'  {a[0]} {a[1]} minus {b_[0]} {b_[1]}: {point:+.3f}  [{lo:+.3f}, {hi:+.3f}]')
    rows.append({'comparison': f'{a[0]} {a[1]} minus {b_[0]} {b_[1]}', 'estimate_pp': point, 'ci_low_pp': lo, 'ci_high_pp': hi})

out_dir = ROOT / 'baseline_sanity_check_20261118'
out_dir.mkdir(exist_ok=True)
pd.DataFrame(rows).to_csv(out_dir / 'bootstrap_intervals.csv', index=False)
print('\nSaved:', out_dir / 'bootstrap_intervals.csv')

95% bootstrap intervals for EER (%)
  original    clean      0.830  [0.734, 0.968]
  original    filtered   4.051  [3.859, 4.270]
  clean_only  clean      0.882  [0.786, 0.954]
  clean_only  filtered   4.025  [3.823, 4.256]
  augmented   clean      1.047  [0.913, 1.183]
  augmented   filtered   3.900  [3.660, 4.095]

Paired EER differences in percentage points (negative means the first model is better)
  augmented filtered minus clean_only filtered: -0.125  [-0.271, -0.015]
  augmented filtered minus original filtered: -0.151  [-0.274, -0.041]
  clean_only filtered minus original filtered: -0.027  [-0.098, +0.056]
  augmented clean minus clean_only clean: +0.165  [+0.098, +0.258]
  augmented clean minus original clean: +0.218  [+0.135, +0.297]

Saved: /content/unpacked_VoiceShield/VoiceShield/baseline_sanity_check_20261118/bootstrap_intervals.csv


In [14]:
# Trivial baselines. Neither one looks at the audio.
train_protocol = DATA / 'ASVspoof2019_LA_cm_protocols' / 'ASVspoof2019.LA.cm.train.trn.txt'
train_labels = [line.split()[-1] for line in train_protocol.read_text().splitlines() if line.strip()]
n_spoof_train, n_genuine_train = train_labels.count('spoof'), train_labels.count('bonafide')
majority_is_genuine = n_genuine_train > n_spoof_train
print(f'Training labels: spoof {n_spoof_train}, genuine {n_genuine_train} -> majority class: '
      f"{'genuine' if majority_is_genuine else 'spoof'}")

n = y.size
is_genuine = (y == 1)

def summarise(name, predicted_genuine, eer=None):
    flagged = int((is_genuine & ~predicted_genuine).sum())
    accepted = int((~is_genuine & predicted_genuine).sum())
    accuracy = (predicted_genuine == is_genuine).mean()
    balanced = (predicted_genuine[is_genuine].mean() + (~predicted_genuine[~is_genuine]).mean()) / 2
    print(f'{name:15s} genuine flagged {flagged}/{is_genuine.sum()} | spoof accepted {accepted}/{(~is_genuine).sum()} '
          f"| accuracy {accuracy*100:.2f}% | balanced accuracy {balanced*100:.2f}%" + (f' | EER {eer*100:.2f}%' if eer is not None else ' | EER omitted (constant scores)'))
    return {'baseline': name, 'genuine_flagged': flagged, 'spoof_accepted': accepted,
            'accuracy': accuracy, 'balanced_accuracy': balanced, 'eer': eer}

majority_pred = np.full(n, majority_is_genuine)
random_scores = np.random.default_rng(42).random(n)     # uniform scores, fixed threshold 0.5
random_pred = random_scores > 0.5
random_eer = eer_fast(random_scores[genuine_idx], random_scores[spoof_idx])

results = [summarise('majority class', majority_pred), summarise('random scores', random_pred, random_eer)]
pd.DataFrame(results).to_csv(out_dir / 'baseline_metrics.csv', index=False)
pd.DataFrame({'recording_id': eval_ids, 'true_label': y, 'majority_pred_genuine': majority_pred.astype(int),
              'random_score': random_scores, 'random_pred_genuine': random_pred.astype(int)}
             ).to_csv(out_dir / 'baseline_predictions.csv', index=False)
(out_dir / 'provenance.json').write_text(json.dumps({
    'numpy': np.__version__, 'random_seed': 42, 'random_threshold': 0.5, 'n_eval': int(n),
    'train_spoof': n_spoof_train, 'train_genuine': n_genuine_train,
    'recording_order': 'order of original_clean_scores.csv'}, indent=2))
print('\nSaved baseline files to', out_dir)
print('Both baselines ignore the audio, so they score the same in clean and filtered conditions.')

Training labels: spoof 22800, genuine 2580 -> majority class: spoof
majority class  genuine flagged 7355/7355 | spoof accepted 0/63882 | accuracy 89.68% | balanced accuracy 50.00% | EER omitted (constant scores)
random scores   genuine flagged 3657/7355 | spoof accepted 32092/63882 | accuracy 49.82% | balanced accuracy 50.02% | EER 49.97%

Saved baseline files to /content/unpacked_VoiceShield/VoiceShield/baseline_sanity_check_20261118
Both baselines ignore the audio, so they score the same in clean and filtered conditions.


### 2.5 Code used for the experiments

The code below covers the subset checks, full baseline, fine-tuning, clean-only control and final evaluation. It is included for inspection and reproduction. **Leave RUN_EXPERIMENTS=False to view the evidence without repeating long runs**

If reproducing, use a separate run, enable the switch and execute C1-C16 in order. C1 creates a folder named with the date and time under BASe, it must run only once for that experiment. C17 packages new results. CPU runs are supported but can be much slower

Nothing below needs running to use the saved model. Colab files under /content are temporary: download any newly created results before disconnecting. The main submitted evidence remains unchanged. Final evaluation must not be used to choose further settings


In [15]:
RUN_EXPERIMENTS = True  # Change only to reproduce the full experiments.


#### C1. Restore the original baseline model
Reload the original pretrained weights and create a separate results folder.


In [16]:
if not RUN_EXPERIMENTS:
    print("Skipped: optional experiment. No model or result files changed.")
else:
    print('Using device:', device)
    from datetime import datetime, timezone
    # Keep new runs separate from the saved evidence.
    ROOT = BASE / 'runs' / datetime.now(timezone.utc).strftime('run_%Y%m%dT%H%M%S%fZ')
    ROOT.mkdir(parents=True, exist_ok=False)
    print('NEW EXPERIMENT OUTPUT FOLDER:', ROOT)
    os.chdir(PROJECT)
    config = json.loads((PROJECT / 'config/AASIST.conf').read_text())
    checkpoint = PROJECT / config['model_path']
    model = Model(config['model_config']).to(device)
    model.load_state_dict(torch.load(checkpoint, map_location=device, weights_only=True), strict=True)
    model.eval()
    RUN_SUBSET_HISTORY = True
    RUN_FULL_BASELINE = True
    # Output folders for reproduced trials are assigned by the training cells below.
    print('Original checkpoint restored for reproduction.')


Using device: cuda
NEW EXPERIMENT OUTPUT FOLDER: /content/runs/run_20261118T064542491901Z
Original checkpoint restored for reproduction.


#### C2. Select the 500-recording development subset
Select 250 genuine and 250 spoof development recordings with seed 42. Save their IDs.


In [17]:
if not RUN_EXPERIMENTS:
    print("Skipped: optional experiment. No model or result files changed.")
else:
    if not RUN_SUBSET_HISTORY:
        print('Skipped completed baseline work. Existing files are unchanged.')
    else:
        import numpy as np
        from pathlib import Path

        # Reuse the development IDs and labels loaded earlier.
        genuine_ids = [uid for uid in ids if labels[uid] == 1]
        spoof_ids = [uid for uid in ids if labels[uid] == 0]

        # A fixed seed makes the selection repeatable.
        rng = np.random.default_rng(42)

        selected_ids = (
            rng.choice(genuine_ids, size=250, replace=False).tolist()
            + rng.choice(spoof_ids, size=250, replace=False).tolist()
        )

        # Confirm the subset is correct.
        assert len(selected_ids) == 500
        assert len(set(selected_ids)) == 500
        assert sum(labels[uid] == 1 for uid in selected_ids) == 250
        assert sum(labels[uid] == 0 for uid in selected_ids) == 250

        # Preserve the exact selection for later comparisons.
        save_path = Path(
            f"{ROOT}/dev_subset_seed42.txt"
        )
        save_path.parent.mkdir(parents=True, exist_ok=True)
        save_path.write_text("\n".join(selected_ids) + "\n")

        print("Selected:", len(selected_ids))
        print("Genuine:", sum(labels[uid] == 1 for uid in selected_ids))
        print("Spoof:", sum(labels[uid] == 0 for uid in selected_ids))
        print("Saved to:", save_path)


Selected: 500
Genuine: 250
Spoof: 250
Saved to: /content/runs/run_20261118T064542491901Z/dev_subset_seed42.txt


#### C3. Score the clean development subset
Score the selected clean recordings with the original model and save one row per recording.


In [18]:
if not RUN_EXPERIMENTS:
    print("Skipped: optional experiment. No model or result files changed.")
else:
    if not RUN_SUBSET_HISTORY:
        print('Skipped completed baseline work. Existing files are unchanged.')
    else:
        import csv
        from pathlib import Path
        import numpy as np
        import torch
        from torch.utils.data import DataLoader
        from data_utils import Dataset_ASVspoof2019_devNeval

        # Check that the selection contains only development recordings.
        assert len(selected_ids) == 500
        assert len(set(selected_ids)) == 500
        assert set(selected_ids).issubset(set(ids))

        dataset = Dataset_ASVspoof2019_devNeval(
            selected_ids,
            DATA / "ASVspoof2019_LA_dev"
        )

        loader = DataLoader(
            dataset,
            batch_size=8,
            shuffle=False,
            num_workers=0
        )

        model.eval()
        rows = []

        # No gradients are calculated and no weights are updated.
        with torch.inference_mode():
            for batch_number, (waveforms, recording_ids) in enumerate(loader):
                waveforms = waveforms.to(device)

                embeddings, logits = model(waveforms)

                # AASIST labels:
                # class 0 = spoof, class 1 = bona fide.
                scores = logits[:, 1].cpu().tolist()

                for uid, score in zip(recording_ids, scores):
                    rows.append({
                        "recording_id": uid,
                        "true_label": int(labels[uid]),
                        "bona_fide_score": float(score)
                    })

                if (batch_number + 1) % 10 == 0:
                    print(f"Scored {len(rows)}/{len(selected_ids)} recordings")

        # Verify completeness, ordering, labels and valid scores.
        assert len(rows) == 500
        assert [row["recording_id"] for row in rows] == selected_ids
        assert len({row["recording_id"] for row in rows}) == 500
        assert all(
            row["true_label"] == labels[row["recording_id"]]
            for row in rows
        )
        assert np.isfinite([row["bona_fide_score"] for row in rows]).all()

        output = Path(
            f"{ROOT}/"
            "aasist_original_dev_subset_clean_scores.csv"
        )
        output.parent.mkdir(parents=True, exist_ok=True)

        with output.open("w", newline="") as stream:
            writer = csv.DictWriter(
                stream,
                fieldnames=["recording_id", "true_label", "bona_fide_score"]
            )
            writer.writeheader()
            writer.writerows(rows)

        print(f"SUCCESS: saved {len(rows)} recording scores.")
        print(output)
        print("First three rows:", rows[:3])


Scored 80/500 recordings
Scored 160/500 recordings
Scored 240/500 recordings
Scored 320/500 recordings
Scored 400/500 recordings
Scored 480/500 recordings
SUCCESS: saved 500 recording scores.
/content/runs/run_20261118T064542491901Z/aasist_original_dev_subset_clean_scores.csv
First three rows: [{'recording_id': 'LA_D_1275044', 'true_label': 1, 'bona_fide_score': 5.720986366271973}, {'recording_id': 'LA_D_4331249', 'true_label': 1, 'bona_fide_score': 5.503896713256836}, {'recording_id': 'LA_D_8163197', 'true_label': 1, 'bona_fide_score': 5.598759174346924}]


#### C4. Calculate subset EER
Calculate subset EER using the official AASIST scoring function.


In [19]:
if not RUN_EXPERIMENTS:
    print("Skipped: optional experiment. No model or result files changed.")
else:
    if not RUN_SUBSET_HISTORY:
        print('Skipped completed baseline work. Existing files are unchanged.')
    else:
        import csv
        import numpy as np
        from evaluation import compute_eer

        score_file = (
            f"{ROOT}/"
            "aasist_original_dev_subset_clean_scores.csv"
        )

        with open(score_file, newline="") as stream:
            results = list(csv.DictReader(stream))

        genuine_scores = np.array([
            float(row["bona_fide_score"])
            for row in results
            if int(row["true_label"]) == 1
        ])

        spoof_scores = np.array([
            float(row["bona_fide_score"])
            for row in results
            if int(row["true_label"]) == 0
        ])

        assert genuine_scores.size > 0 and spoof_scores.size > 0
        assert np.isfinite(genuine_scores).all()
        assert np.isfinite(spoof_scores).all()

        eer, eer_threshold = compute_eer(genuine_scores, spoof_scores)

        print(f"Genuine recordings: {len(genuine_scores)}")
        print(f"Spoof recordings: {len(spoof_scores)}")
        print(f"Development subset EER: {eer * 100:.2f}%")
        print(f"EER threshold: {eer_threshold:.6f}")


Genuine recordings: 250
Spoof recordings: 250
Development subset EER: 0.40%
EER threshold: -2.414625


#### C5. Filter and score the same subset
Apply the 300-3400 Hz filter to both classes before scoring the same recordings.


In [20]:
if not RUN_EXPERIMENTS:
    print("Skipped: optional experiment. No model or result files changed.")
else:
    if not RUN_SUBSET_HISTORY:
        print('Skipped completed baseline work. Existing files are unchanged.')
    else:
        import csv
        from pathlib import Path

        import numpy as np
        import soundfile as sf
        import torch
        from scipy.signal import butter, sosfilt
        from torch.utils.data import Dataset, DataLoader

        from data_utils import pad
        from evaluation import compute_eer

        # Read the exact recording selection and labels from your clean results.
        result_folder = Path(f"{ROOT}")
        clean_file = result_folder / "aasist_original_dev_subset_clean_scores.csv"

        with clean_file.open(newline="") as stream:
            clean_rows = list(csv.DictReader(stream))

        recording_ids = [row["recording_id"] for row in clean_rows]
        recording_labels = {
            row["recording_id"]: int(row["true_label"])
            for row in clean_rows
        }

        assert len(recording_ids) == len(set(recording_ids)) == 500

        # Fixed, deterministic band-pass filter.
        telephone_filter = butter(
            N=6,
            Wn=[300, 3400],
            btype="bandpass",
            fs=16000,
            output="sos"
        )


        class BandLimitedDevelopment(Dataset):
            def __len__(self):
                return len(recording_ids)

            def __getitem__(self, index):
                uid = recording_ids[index]
                path = DATA / "ASVspoof2019_LA_dev" / "flac" / f"{uid}.flac"

                waveform, sample_rate = sf.read(path)

                assert sample_rate == 16000
                assert waveform.ndim == 1 and waveform.size > 0
                assert np.isfinite(waveform).all()

                # Apply the same transformation to genuine and spoof recordings.
                filtered = sosfilt(telephone_filter, waveform)

                # Preserve the official AASIST input-length policy.
                prepared = pad(filtered, max_len=64600)
                prepared = np.asarray(prepared, dtype=np.float32)

                assert prepared.shape == (64600,)
                assert np.isfinite(prepared).all()

                return torch.from_numpy(prepared), uid


        loader = DataLoader(
            BandLimitedDevelopment(),
            batch_size=8,
            shuffle=False,
            num_workers=0
        )

        model.eval()
        degraded_rows = []

        with torch.inference_mode():
            for waveforms, batch_ids in loader:
                _, logits = model(waveforms.to(device))
                scores = logits[:, 1].cpu().tolist()

                for uid, score in zip(batch_ids, scores):
                    degraded_rows.append({
                        "recording_id": uid,
                        "true_label": recording_labels[uid],
                        "bona_fide_score": float(score)
                    })

        assert [r["recording_id"] for r in degraded_rows] == recording_ids
        assert np.isfinite([
            r["bona_fide_score"] for r in degraded_rows
        ]).all()

        degraded_file = (
            result_folder / "aasist_original_dev_subset_bandlimited_scores.csv"
        )

        with degraded_file.open("w", newline="") as stream:
            writer = csv.DictWriter(
                stream,
                fieldnames=["recording_id", "true_label", "bona_fide_score"]
            )
            writer.writeheader()
            writer.writerows(degraded_rows)

        # Calculate EER separately for each condition.
        def condition_eer(records):
            genuine = np.array([
                float(r["bona_fide_score"])
                for r in records if int(r["true_label"]) == 1
            ])
            spoof = np.array([
                float(r["bona_fide_score"])
                for r in records if int(r["true_label"]) == 0
            ])
            return compute_eer(genuine, spoof)


        clean_eer, clean_threshold = condition_eer(clean_rows)
        bandlimited_eer, _ = condition_eer(degraded_rows)

        print(f"Recordings scored: {len(degraded_rows)}")
        print(f"Clean EER: {clean_eer * 100:.2f}%")
        print(f"Band-limited EER: {bandlimited_eer * 100:.2f}%")
        print(
            "EER change:",
            f"{(bandlimited_eer - clean_eer) * 100:+.2f} percentage points"
        )
        print(f"Clean development threshold retained: {clean_threshold:.6f}")
        print("Saved:", degraded_file)


Recordings scored: 500
Clean EER: 0.40%
Band-limited EER: 1.20%
EER change: +0.80 percentage points
Clean development threshold retained: -2.414625
Saved: /content/runs/run_20261118T064542491901Z/aasist_original_dev_subset_bandlimited_scores.csv


#### C6. Check fixed-threshold subset errors
Keep the clean-development threshold fixed when measuring filtered error rates.


In [21]:
if not RUN_EXPERIMENTS:
    print("Skipped: optional experiment. No model or result files changed.")
else:
    if not RUN_SUBSET_HISTORY:
        print('Skipped completed baseline work. Existing files are unchanged.')
    else:
        import csv
        from pathlib import Path
        import numpy as np
        from evaluation import compute_eer

        folder = Path(f"{ROOT}")

        def read_scores(filename):
            with (folder / filename).open(newline="") as stream:
                records = list(csv.DictReader(stream))

            ids = [r["recording_id"] for r in records]
            labels = np.array([int(r["true_label"]) for r in records])
            scores = np.array([float(r["bona_fide_score"]) for r in records])

            assert len(ids) == len(set(ids))
            assert set(labels.tolist()) == {0, 1}
            assert np.isfinite(scores).all()

            return ids, labels, scores


        clean_ids, clean_labels, clean_scores = read_scores(
            "aasist_original_dev_subset_clean_scores.csv"
        )

        degraded_ids, degraded_labels, degraded_scores = read_scores(
            "aasist_original_dev_subset_bandlimited_scores.csv"
        )

        # Verify that both conditions contain identical recordings and labels.
        assert clean_ids == degraded_ids
        np.testing.assert_array_equal(clean_labels, degraded_labels)

        # Choose the threshold ONLY from clean development scores.
        _, threshold = compute_eer(
            clean_scores[clean_labels == 1],
            clean_scores[clean_labels == 0]
        )

        print(f"Fixed clean-development threshold: {threshold:.9f}\n")

        summary = []

        for condition, scores in [
            ("Clean", clean_scores),
            ("Band-limited", degraded_scores)
        ]:
            # Higher scores indicate genuine speech.
            # Scores equal to the threshold are classified as spoof.
            predicted_genuine = scores > threshold

            genuine = clean_labels == 1
            spoof = clean_labels == 0

            genuine_flagged = int(np.sum(genuine & ~predicted_genuine))
            spoof_missed = int(np.sum(spoof & predicted_genuine))

            genuine_total = int(genuine.sum())
            spoof_total = int(spoof.sum())

            false_alarm_rate = genuine_flagged / genuine_total
            miss_rate = spoof_missed / spoof_total

            print(condition)
            print(
                f"  Genuine incorrectly flagged: "
                f"{genuine_flagged}/{genuine_total} "
                f"({false_alarm_rate:.2%})"
            )
            print(
                f"  Spoof incorrectly accepted: "
                f"{spoof_missed}/{spoof_total} "
                f"({miss_rate:.2%})\n"
            )

            summary.append({
                "condition": condition,
                "threshold": float(threshold),
                "genuine_total": genuine_total,
                "genuine_flagged_as_spoof": genuine_flagged,
                "genuine_false_alarm_rate": false_alarm_rate,
                "spoof_total": spoof_total,
                "spoof_accepted_as_genuine": spoof_missed,
                "spoof_miss_rate": miss_rate
            })

        output = folder / "aasist_original_dev_subset_fixed_threshold_metrics.csv"

        with output.open("w", newline="") as stream:
            writer = csv.DictWriter(stream, fieldnames=list(summary[0]))
            writer.writeheader()
            writer.writerows(summary)

        print("Saved:", output)


Fixed clean-development threshold: -2.414624691

Clean
  Genuine incorrectly flagged: 1/250 (0.40%)
  Spoof incorrectly accepted: 1/250 (0.40%)

Band-limited
  Genuine incorrectly flagged: 3/250 (1.20%)
  Spoof incorrectly accepted: 3/250 (1.20%)

Saved: /content/runs/run_20261118T064542491901Z/aasist_original_dev_subset_fixed_threshold_metrics.csv


#### C7. Score the complete development baseline
Repeat the baseline on all 24,844 development recordings in both conditions.


In [22]:
if not RUN_EXPERIMENTS:
    print("Skipped: optional experiment. No model or result files changed.")
else:
    if not RUN_FULL_BASELINE:
        print('Skipped completed baseline work. Existing files are unchanged.')
    else:
        import csv
        from pathlib import Path

        import numpy as np
        import soundfile as sf
        import torch
        from scipy.signal import butter, sosfilt
        from torch.utils.data import Dataset, DataLoader

        from data_utils import genSpoof_list, pad
        from evaluation import compute_eer

        folder = Path(f"{ROOT}")
        folder.mkdir(parents=True, exist_ok=True)

        protocol = (
            DATA / "ASVspoof2019_LA_cm_protocols"
            / "ASVspoof2019.LA.cm.dev.trl.txt"
        )

        dev_labels, dev_ids = genSpoof_list(
            str(protocol), is_train=False, is_eval=False
        )

        assert len(dev_ids) == len(set(dev_ids))
        print("Development recordings:", len(dev_ids))

        telephone_filter = butter(
            6, [300, 3400],
            btype="bandpass",
            fs=16000,
            output="sos"
        )


        class FullDevelopment(Dataset):
            def __init__(self, bandlimited=False):
                self.bandlimited = bandlimited

            def __len__(self):
                return len(dev_ids)

            def __getitem__(self, index):
                uid = dev_ids[index]
                path = DATA / "ASVspoof2019_LA_dev/flac" / f"{uid}.flac"

                waveform, sr = sf.read(path)
                assert sr == 16000
                assert waveform.ndim == 1 and waveform.size > 0
                assert np.isfinite(waveform).all()

                if self.bandlimited:
                    waveform = sosfilt(telephone_filter, waveform)

                waveform = np.asarray(
                    pad(waveform, max_len=64600),
                    dtype=np.float32
                )
                assert waveform.shape == (64600,)
                assert np.isfinite(waveform).all()

                return torch.from_numpy(waveform), uid


        def score_condition(condition, bandlimited):
            destination = folder / (
                f"aasist_original_dev_full_{condition}_scores.csv"
            )

            # Reuse a completed result when rerunning this cell.
            if destination.is_file():
                with destination.open(newline="") as stream:
                    records = list(csv.DictReader(stream))
                print("Loading existing:", destination.name)
            else:
                loader = DataLoader(
                    FullDevelopment(bandlimited),
                    batch_size=8,
                    shuffle=False,
                    num_workers=0
                )

                records = []
                model.eval()

                with torch.inference_mode():
                    for batch_number, (waveforms, batch_ids) in enumerate(loader):
                        _, logits = model(waveforms.to(device))
                        scores = logits[:, 1].cpu().tolist()

                        for uid, score in zip(batch_ids, scores):
                            records.append({
                                "recording_id": uid,
                                "true_label": int(dev_labels[uid]),
                                "bona_fide_score": float(score)
                            })

                        if (batch_number + 1) % 100 == 0:
                            print(
                                f"{condition}: "
                                f"{len(records)}/{len(dev_ids)}"
                            )

            assert [r["recording_id"] for r in records] == dev_ids
            assert all(
                int(r["true_label"]) == dev_labels[r["recording_id"]]
                for r in records
            )

            scores = np.array([
                float(r["bona_fide_score"]) for r in records
            ])
            assert np.isfinite(scores).all()

            # Save only after completeness checks pass.
            with destination.open("w", newline="") as stream:
                writer = csv.DictWriter(
                    stream,
                    fieldnames=[
                        "recording_id", "true_label", "bona_fide_score"
                    ]
                )
                writer.writeheader()
                writer.writerows(records)

            print("Completed:", condition)
            return scores


        clean_scores_full = score_condition("clean", False)
        filtered_scores_full = score_condition("bandlimited", True)

        y = np.array([dev_labels[uid] for uid in dev_ids])
        genuine = y == 1
        spoof = y == 0

        clean_eer_full, clean_threshold_full = compute_eer(
            clean_scores_full[genuine],
            clean_scores_full[spoof]
        )

        summary = []

        for condition, scores in [
            ("clean", clean_scores_full),
            ("bandlimited", filtered_scores_full)
        ]:
            eer, _ = compute_eer(scores[genuine], scores[spoof])

            # Apply the same CLEAN threshold in both conditions.
            predicted_genuine = scores > clean_threshold_full

            genuine_flagged = int(np.sum(genuine & ~predicted_genuine))
            spoof_missed = int(np.sum(spoof & predicted_genuine))

            false_alarm_rate = genuine_flagged / int(genuine.sum())
            miss_rate = spoof_missed / int(spoof.sum())

            print(f"\n{condition.upper()}")
            print(f"EER: {eer * 100:.3f}%")
            print(
                f"Genuine flagged: {genuine_flagged}/{genuine.sum()} "
                f"({false_alarm_rate:.3%})"
            )
            print(
                f"Spoof accepted: {spoof_missed}/{spoof.sum()} "
                f"({miss_rate:.3%})"
            )

            summary.append({
                "condition": condition,
                "eer": float(eer),
                "clean_development_threshold": float(clean_threshold_full),
                "genuine_total": int(genuine.sum()),
                "genuine_flagged": genuine_flagged,
                "genuine_false_alarm_rate": false_alarm_rate,
                "spoof_total": int(spoof.sum()),
                "spoof_missed": spoof_missed,
                "spoof_miss_rate": miss_rate
            })

        summary_file = folder / "aasist_original_dev_full_metrics.csv"
        with summary_file.open("w", newline="") as stream:
            writer = csv.DictWriter(stream, fieldnames=list(summary[0]))
            writer.writeheader()
            writer.writerows(summary)

        print(f"\nFixed clean threshold: {clean_threshold_full:.9f}")
        print("Metrics saved:", summary_file)


Development recordings: 24844
clean: 800/24844
clean: 1600/24844
clean: 2400/24844
clean: 3200/24844


KeyboardInterrupt: 

#### C8. Read the training protocol
Read the training labels without changing the model.


In [23]:
if not RUN_EXPERIMENTS:
    print("Skipped: optional experiment. No model or result files changed.")
else:
    from pathlib import Path
    from data_utils import genSpoof_list

    train_protocol = (
        DATA / 'ASVspoof2019_LA_cm_protocols'
        / 'ASVspoof2019.LA.cm.train.trn.txt'
    )
    assert train_protocol.is_file(), f'Missing training protocol: {train_protocol}'
    train_labels, train_ids = genSpoof_list(
        str(train_protocol), is_train=True, is_eval=False
    )
    assert len(train_ids) == len(set(train_ids)), 'Duplicate training recording IDs'
    assert set(train_labels.values()) == {0, 1}, 'Expected spoof=0 and genuine=1'
    print('Training recordings:', len(train_ids))
    print('Genuine:', sum(train_labels[uid] == 1 for uid in train_ids))
    print('Spoof:', sum(train_labels[uid] == 0 for uid in train_ids))


Training recordings: 25380
Genuine: 2580
Spoof: 22800


#### C9. Select and verify the training subset
Save 4,000 training IDs and check audio availability and development-ID separation.


In [24]:
if not RUN_EXPERIMENTS:
    print("Skipped: optional experiment. No model or result files changed.")
else:
    from pathlib import Path
    import numpy as np
    from data_utils import genSpoof_list

    subset_path = Path(f'{ROOT}/train_subset_4000_seed42.txt')
    if subset_path.is_file():
        training_subset_ids = subset_path.read_text().splitlines()
        print('Reusing saved training subset.')
    else:
        rng = np.random.default_rng(42)
        genuine_train = [uid for uid in train_ids if train_labels[uid] == 1]
        spoof_train = [uid for uid in train_ids if train_labels[uid] == 0]
        training_subset_ids = (
            rng.choice(genuine_train, 407, replace=False).tolist()
            + rng.choice(spoof_train, 3593, replace=False).tolist()
        )
        rng.shuffle(training_subset_ids)

    assert len(training_subset_ids) == len(set(training_subset_ids)) == 4000
    assert set(training_subset_ids).issubset(set(train_ids))
    assert sum(train_labels[uid] == 1 for uid in training_subset_ids) == 407
    assert sum(train_labels[uid] == 0 for uid in training_subset_ids) == 3593

    dev_protocol = DATA / 'ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.dev.trl.txt'
    _, development_ids = genSpoof_list(str(dev_protocol), is_train=False, is_eval=False)
    assert not set(training_subset_ids).intersection(development_ids), 'Training/development ID overlap'

    train_audio_folder = DATA / 'ASVspoof2019_LA_train/flac'
    missing = []
    for uid in training_subset_ids:
        path = train_audio_folder / f'{uid}.flac'
        if not path.is_file() or path.stat().st_size == 0:
            missing.append(uid)
    assert not missing, f'Missing or empty training audio (first 10): {missing[:10]}'

    if not subset_path.exists():
        subset_path.parent.mkdir(parents=True, exist_ok=True)
        with subset_path.open('x') as stream:
            stream.write('\n'.join(training_subset_ids) + '\n')

    print('Training subset: 4000')
    print('Genuine: 407')
    print('Spoof: 3593')
    print('Missing or empty audio: 0')
    print('Development recording-ID overlap: 0')
    print('Saved selection:', subset_path)
    print('Model unchanged. Ready to plan the fine-tuning trial.')


Training subset: 4000
Genuine: 407
Spoof: 3593
Missing or empty audio: 0
Development recording-ID overlap: 0
Saved selection: /content/runs/run_20261118T064542491901Z/train_subset_4000_seed42.txt
Model unchanged. Ready to plan the fine-tuning trial.


#### C10. Check training shapes, loss and gradients
Check an eight-recording batch, loss and gradients without updating weights. BatchNorm statistics are held fixed for this diagnostic.


In [25]:
if not RUN_EXPERIMENTS:
    print("Skipped: optional experiment. No model or result files changed.")
else:
    import random
    import numpy as np
    import soundfile as sf
    import torch
    from scipy.signal import butter, sosfilt
    from data_utils import pad
    from models.AASIST import Model

    print('Using device:', device)
    random.seed(42)
    np.random.seed(42)
    torch.manual_seed(42)
    if device.type == 'cuda':
        torch.cuda.manual_seed_all(42)
    # Use a fresh model object, never the original baseline object.
    trial_model = Model(config['model_config']).to(device)
    trial_model.load_state_dict(torch.load(checkpoint, map_location=device, weights_only=True), strict=True)
    trial_model.train()
    for layer in trial_model.modules():
        if isinstance(layer, torch.nn.modules.batchnorm._BatchNorm):
            layer.eval()  # avoid changing running statistics during this diagnostic

    batch_ids = (
        [uid for uid in training_subset_ids if train_labels[uid] == 1][:4]
        + [uid for uid in training_subset_ids if train_labels[uid] == 0][:4]
    )
    bandpass = butter(6, [300, 3400], btype='bandpass', fs=16000, output='sos')
    examples = []
    for i, uid in enumerate(batch_ids):
        waveform, sr = sf.read(train_audio_folder / f'{uid}.flac')
        assert sr == 16000 and waveform.ndim == 1 and waveform.size > 0
        assert np.isfinite(waveform).all()
        if i % 2 == 0:
            waveform = sosfilt(bandpass, waveform)
        examples.append(np.asarray(pad(waveform, max_len=64600), dtype=np.float32))

    batch_x = torch.from_numpy(np.stack(examples)).to(device)
    batch_y = torch.tensor([train_labels[uid] for uid in batch_ids], dtype=torch.long, device=device)
    assert tuple(batch_x.shape) == (8, 64600)
    assert torch.isfinite(batch_x).all()
    trial_model.zero_grad(set_to_none=True)
    _, batch_logits = trial_model(batch_x)
    assert tuple(batch_logits.shape) == (8, 2)
    assert torch.isfinite(batch_logits).all()
    # Match the class-weight convention in the official AASIST training code.
    loss_fn = torch.nn.CrossEntropyLoss(weight=torch.tensor([0.1, 0.9], device=device))
    loss = loss_fn(batch_logits, batch_y)
    assert torch.isfinite(loss)
    loss.backward()
    gradients = [p.grad for p in trial_model.parameters() if p.grad is not None]
    assert gradients and all(torch.isfinite(g).all() for g in gradients)
    assert any(torch.count_nonzero(g).item() > 0 for g in gradients)
    print('PASS: input shape', tuple(batch_x.shape))
    print('PASS: output shape', tuple(batch_logits.shape))
    print('Batch labels:', batch_y.tolist())
    print('Loss:', loss.item())
    print('PASS: gradients are finite and not all zero.')
    print('No weights updated. Original baseline model was not changed.')
    trial_model.zero_grad(set_to_none=True)
    del trial_model, batch_x, batch_y, batch_logits, loss, gradients
    if device.type == 'cuda':
        torch.cuda.empty_cache()

Using device: cuda
PASS: input shape (8, 64600)
PASS: output shape (8, 2)
Batch labels: [1, 1, 1, 1, 0, 0, 0, 0]
Loss: 0.22896303236484528
PASS: gradients are finite and not all zero.
No weights updated. Original baseline model was not changed.


#### C11. Fine-tune with bandwidth augmentation
Start from the original weights. Fine-tune one epoch with random bandwidth filtering applied with probability 0.5. BatchNorm runs in training mode.


In [27]:
if not RUN_EXPERIMENTS:
    print("Skipped: optional experiment. No model or result files changed.")
else:
    # One additional epoch from pretrained weights.
    import json, random, time, hashlib, subprocess, os
    from datetime import datetime, timezone
    from pathlib import Path
    import numpy as np
    import soundfile as sf
    import torch
    from torch.utils.data import Dataset, DataLoader
    from scipy.signal import butter, sosfilt
    from models.AASIST import Model
    from data_utils import pad

    print('Using device:', device)
    SEED = 42
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if device.type == 'cuda':
        torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    # Keep random choices repeatable within this environment.

    training_ids = list(training_subset_ids)
    assert len(training_ids) == len(set(training_ids)) == 4000
    assert set(training_ids).issubset(train_labels)
    assert not set(training_ids).intersection(development_ids)

    run_dir = Path(f'{ROOT}') / (
        'adaptation_trial_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    )
    run_dir.mkdir(parents=True, exist_ok=False)
    settings = {
        'seed': SEED, 'epochs': 1, 'batch_size': 8, 'optimizer': 'Adam',
        'learning_rate': 1e-6, 'weight_decay': 0.0, 'loss_class_weights_spoof_genuine': [0.1, 0.9],
        'augmentation_probability': 0.5, 'filter': 'butter N=6, 300-3400 Hz, SOS, sosfilt',
        'sample_rate': 16000, 'samples': 64600, 'crop': 'random training crop; repeat short audio',
        'batchnorm': 'training mode; running statistics updated',
        'checkpoint_sha256': hashlib.sha256(checkpoint.read_bytes()).hexdigest(),
        'source_commit': subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip(),
        'torch': torch.__version__, 'numpy': np.__version__,
        'gpu': torch.cuda.get_device_name(0) if device.type == 'cuda' else None,
        'model_config': config['model_config'],
        'selection': 'fixed one-epoch trial; no best-checkpoint claim'
    }
    (run_dir / 'settings.json').write_text(json.dumps(settings, indent=2))
    (run_dir / 'training_ids.txt').write_text('\n'.join(training_ids) + '\n')

    class AdaptationTraining(Dataset):
        def __init__(self):
            self.rng = np.random.default_rng(SEED)
            self.sos = butter(6, [300, 3400], btype='bandpass', fs=16000, output='sos')
            self.filtered_count = 0

        def __len__(self):
            return len(training_ids)

        def __getitem__(self, i):
            uid = training_ids[i]
            y, sr = sf.read(DATA / 'ASVspoof2019_LA_train/flac' / f'{uid}.flac')
            assert sr == 16000 and y.ndim == 1 and y.size > 0 and np.isfinite(y).all()
            if self.rng.random() < 0.5:
                y = sosfilt(self.sos, y)
                self.filtered_count += 1
            if len(y) > 64600:
                start = int(self.rng.integers(0, len(y) - 64600 + 1))
                y = y[start:start + 64600]
            else:
                y = pad(y, max_len=64600)
            x = torch.from_numpy(np.asarray(y, dtype=np.float32))
            assert torch.isfinite(x).all()
            return x, train_labels[uid]

    training_data = AdaptationTraining()
    training_loader = DataLoader(training_data, batch_size=8, shuffle=True, num_workers=0,
                                generator=torch.Generator().manual_seed(SEED))
    # Start this run from the original checkpoint, not a previous fine-tuned model.
    adapted_model = Model(config['model_config']).to(device)
    adapted_model.load_state_dict(torch.load(checkpoint, map_location=device, weights_only=True), strict=True)
    adapted_model.train()
    optimizer = torch.optim.Adam(adapted_model.parameters(), lr=1e-6)
    criterion = torch.nn.CrossEntropyLoss(weight=torch.tensor([0.1, 0.9], device=device))
    loss_sum, seen = 0.0, 0
    if device.type == 'cuda':
        torch.cuda.synchronize()
    started = time.perf_counter()
    print('Training started. Output folder:', run_dir)
    for batch_number, (x, y) in enumerate(training_loader, start=1):
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad(set_to_none=True)
        _, logits = adapted_model(x)
        loss = criterion(logits, y)
        assert torch.isfinite(loss), 'Nonfinite training loss; stop and investigate.'
        loss.backward()
        for parameter in adapted_model.parameters():
            if parameter.grad is not None:
                assert torch.isfinite(parameter.grad).all(), 'Nonfinite gradient'
        optimizer.step()
        seen += len(y)
        loss_sum += loss.item() * len(y)
        if batch_number % 50 == 0:
            print(f'Batch {batch_number}/{len(training_loader)} | recordings {seen}/4000 | loss {loss.item():.4f}')

    assert seen == 4000
    if device.type == 'cuda':
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - started
    adapted_model.eval()
    # Atomic replacement prevents a partial file being mistaken for a finished checkpoint.
    temporary = run_dir / 'adapted_epoch1.pth.partial'
    torch.save(adapted_model.state_dict(), temporary)
    os.replace(temporary, run_dir / 'adapted_epoch1.pth')
    training_log = {'epoch': 1, 'recordings': seen, 'filtered_recordings': training_data.filtered_count,
                    'batch_loss_mean_weighted_by_batch_size': loss_sum / seen, 'training_seconds': elapsed}
    (run_dir / 'training_log.json').write_text(json.dumps(training_log, indent=2))
    print('TRAINING COMPLETE:', training_log)
    print('Checkpoint:', run_dir / 'adapted_epoch1.pth')
    print('Original model unchanged. Adapted-model development evaluation is still pending.')

    augmented_trial_dir = run_dir


Using device: cuda
Training started. Output folder: /content/runs/run_20261118T064542491901Z/adaptation_trial_20261118T064806504792Z
Batch 50/500 | recordings 400/4000 | loss 0.0005


KeyboardInterrupt: 

#### C12. Evaluate the augmented model on development data
Reload the augmented checkpoint and score clean and filtered development audio. Choose its threshold from clean development only.


In [31]:
if not RUN_EXPERIMENTS:
    print("Skipped: optional experiment. No model or result files changed.")
else:
    import csv, json, hashlib
    from pathlib import Path
    import numpy as np
    import soundfile as sf
    import torch
    from torch.utils.data import Dataset, DataLoader
    from scipy.signal import butter, sosfilt
    from models.AASIST import Model
    from data_utils import genSpoof_list, pad
    from evaluation import compute_eer

    # Evaluate this completed trial explicitly; do not rerun training.
    trial_dir = Path(augmented_trial_dir)
    adapted_checkpoint = trial_dir / 'adapted_epoch1.pth'
    assert adapted_checkpoint.is_file(), f'Missing checkpoint: {adapted_checkpoint}'
    print('Using device:', device)
    settings = json.loads((trial_dir / 'settings.json').read_text())
    protocol = DATA / 'ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.dev.trl.txt'
    eval_dev_labels, eval_dev_ids = genSpoof_list(str(protocol), is_train=False, is_eval=False)
    assert len(eval_dev_ids) == len(set(eval_dev_ids)) == 24844

    # Bind cached scores to this exact checkpoint and evaluation policy.
    provenance = {
        'checkpoint_sha256': hashlib.sha256(adapted_checkpoint.read_bytes()).hexdigest(),
        'protocol_sha256': hashlib.sha256(protocol.read_bytes()).hexdigest(),
        'crop': 'first 64600 samples; repeat short audio',
        'filter': 'butter N=6, 300-3400 Hz, SOS, sosfilt before crop',
        'score': 'class-1 raw bona-fide logit'
    }
    metadata_file = trial_dir / 'development_scoring_settings.json'
    if metadata_file.exists():
        assert json.loads(metadata_file.read_text()) == provenance, 'Cached experiment settings differ'
    else:
        assert not list(trial_dir.glob('adapted_dev_*_scores.csv')), 'Existing scores lack provenance'
        metadata_file.write_text(json.dumps(provenance, indent=2))

    # Reload the saved adapted weights; leave the baseline model untouched.
    evaluation_model = Model(settings['model_config']).to(device)
    evaluation_model.load_state_dict(torch.load(adapted_checkpoint, map_location=device, weights_only=True), strict=True)
    evaluation_model.eval()

    class AdaptedDevelopment(Dataset):
        def __init__(self, filtered):
            self.filtered = filtered
            self.sos = butter(6, [300, 3400], btype='bandpass', fs=16000, output='sos')
        def __len__(self):
            return len(eval_dev_ids)
        def __getitem__(self, i):
            uid = eval_dev_ids[i]
            y, sr = sf.read(DATA / 'ASVspoof2019_LA_dev/flac' / f'{uid}.flac')
            assert sr == 16000 and y.ndim == 1 and y.size > 0 and np.isfinite(y).all()
            if self.filtered:
                y = sosfilt(self.sos, y)
            x = torch.from_numpy(np.asarray(pad(y, max_len=64600), dtype=np.float32))
            assert x.shape == (64600,) and torch.isfinite(x).all()
            return x, uid

    def adapted_scores(condition, filtered):
        path = trial_dir / f'adapted_dev_{condition}_scores.csv'
        if path.exists():
            with path.open(newline='') as stream:
                records = list(csv.DictReader(stream))
            print('Reusing completed:', path.name)
        else:
            records = []
            batches = DataLoader(AdaptedDevelopment(filtered), batch_size=8, shuffle=False, num_workers=0)
            with torch.inference_mode():
                for i, (x, uids) in enumerate(batches, start=1):
                    _, logits = evaluation_model(x.to(device))
                    for uid, score in zip(uids, logits[:, 1].cpu().tolist()):
                        records.append({'recording_id': uid, 'true_label': int(eval_dev_labels[uid]),
                                        'bona_fide_score': float(score)})
                    if i % 100 == 0:
                        print(f'{condition}: {len(records)}/24844')
        assert [r['recording_id'] for r in records] == eval_dev_ids
        assert all(int(r['true_label']) == eval_dev_labels[r['recording_id']] for r in records)
        scores = np.array([float(r['bona_fide_score']) for r in records])
        assert np.isfinite(scores).all()
        if not path.exists():
            temporary = path.with_suffix('.csv.partial')
            with temporary.open('w', newline='') as stream:
                writer = csv.DictWriter(stream, fieldnames=['recording_id', 'true_label', 'bona_fide_score'])
                writer.writeheader()
                writer.writerows(records)
            temporary.replace(path)
        print('Completed:', condition)
        return scores

    adapted_clean_scores = adapted_scores('clean', False)
    adapted_filtered_scores = adapted_scores('bandlimited', True)
    y = np.array([eval_dev_labels[uid] for uid in eval_dev_ids])
    genuine, spoof = y == 1, y == 0
    _, adapted_threshold = compute_eer(adapted_clean_scores[genuine], adapted_clean_scores[spoof])
    adapted_metrics = []
    for condition, scores in [('clean', adapted_clean_scores), ('bandlimited', adapted_filtered_scores)]:
        eer, _ = compute_eer(scores[genuine], scores[spoof])
        predicted_genuine = scores > adapted_threshold
        flagged = int(np.sum(genuine & ~predicted_genuine))
        missed = int(np.sum(spoof & predicted_genuine))
        record = {'condition': condition, 'eer': float(eer),
                  'clean_development_threshold': float(adapted_threshold),
                  'genuine_total': int(genuine.sum()), 'genuine_flagged': flagged,
                  'genuine_false_alarm_rate': flagged / int(genuine.sum()),
                  'spoof_total': int(spoof.sum()), 'spoof_missed': missed,
                  'spoof_miss_rate': missed / int(spoof.sum())}
        adapted_metrics.append(record)
        print(f'ADAPTED {condition}: EER {eer*100:.3f}% | genuine flagged {flagged}/{genuine.sum()} | spoof accepted {missed}/{spoof.sum()}')
    metrics_path = trial_dir / 'adapted_dev_metrics.csv'
    with metrics_path.open('w', newline='') as stream:
        writer = csv.DictWriter(stream, fieldnames=list(adapted_metrics[0]))
        writer.writeheader()
        writer.writerows(adapted_metrics)

    baseline_path = trial_dir.parent / 'aasist_original_dev_full_metrics.csv'
    if baseline_path.exists():
        with baseline_path.open(newline='') as stream:
            baseline = {r['condition']: r for r in csv.DictReader(stream)}
        print('Comparison (negative EER change means improvement):')
        for r in adapted_metrics:
            original_eer = float(baseline[r['condition']]['eer'])
            print(f"{r['condition']}: original {original_eer*100:.3f}% -> adapted {r['eer']*100:.3f}% | change {(r['eer']-original_eer)*100:+.3f} percentage points")
    print('Adapted clean development threshold:', adapted_threshold)
    print('Metrics saved:', metrics_path)
    print('Development evaluation complete. Final evaluation partition remains untouched.')


NameError: name 'augmented_trial_dir' is not defined

#### C13. Fine-tune the clean-only control
Repeat the same training budget from the original weights without filtering. Keep the same subset, seed and random-crop decisions.


In [32]:
if not RUN_EXPERIMENTS:
    print("Skipped: optional experiment. No model or result files changed.")
else:
    # One additional epoch from pretrained weights.
    import json, random, time, hashlib, subprocess, os
    from datetime import datetime, timezone
    from pathlib import Path
    import numpy as np
    import soundfile as sf
    import torch
    from torch.utils.data import Dataset, DataLoader
    from scipy.signal import butter, sosfilt
    from models.AASIST import Model
    from data_utils import pad

    print('Using device:', device)
    SEED = 42
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if device.type == 'cuda':
        torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    # Keep random choices repeatable within this environment.

    training_ids = list(training_subset_ids)
    assert len(training_ids) == len(set(training_ids)) == 4000
    assert set(training_ids).issubset(train_labels)
    assert not set(training_ids).intersection(development_ids)

    run_dir = Path(f'{ROOT}') / (
        'clean_control_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    )
    run_dir.mkdir(parents=True, exist_ok=False)
    settings = {
        'seed': SEED, 'epochs': 1, 'batch_size': 8, 'optimizer': 'Adam',
        'learning_rate': 1e-6, 'weight_decay': 0.0, 'loss_class_weights_spoof_genuine': [0.1, 0.9],
        'augmentation_probability': 0.0, 'filter': 'none; clean-only training',
        'sample_rate': 16000, 'samples': 64600, 'crop': 'random training crop; repeat short audio',
        'batchnorm': 'training mode; running statistics updated',
        'checkpoint_sha256': hashlib.sha256(checkpoint.read_bytes()).hexdigest(),
        'source_commit': subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip(),
        'torch': torch.__version__, 'numpy': np.__version__,
        'gpu': torch.cuda.get_device_name(0) if device.type == 'cuda' else None,
        'model_config': config['model_config'],
        'selection': 'fixed one-epoch trial; no best-checkpoint claim'
    }
    (run_dir / 'settings.json').write_text(json.dumps(settings, indent=2))
    (run_dir / 'training_ids.txt').write_text('\n'.join(training_ids) + '\n')

    class AdaptationTraining(Dataset):
        def __init__(self):
            self.rng = np.random.default_rng(SEED)
            self.sos = butter(6, [300, 3400], btype='bandpass', fs=16000, output='sos')
            self.filtered_count = 0

        def __len__(self):
            return len(training_ids)

        def __getitem__(self, i):
            uid = training_ids[i]
            y, sr = sf.read(DATA / 'ASVspoof2019_LA_train/flac' / f'{uid}.flac')
            assert sr == 16000 and y.ndim == 1 and y.size > 0 and np.isfinite(y).all()
            apply_filter_draw = self.rng.random()  # consume the same draw as the augmented run
            if apply_filter_draw < 0.0:
                y = sosfilt(self.sos, y)
                self.filtered_count += 1
            if len(y) > 64600:
                start = int(self.rng.integers(0, len(y) - 64600 + 1))
                y = y[start:start + 64600]
            else:
                y = pad(y, max_len=64600)
            x = torch.from_numpy(np.asarray(y, dtype=np.float32))
            assert torch.isfinite(x).all()
            return x, train_labels[uid]

    training_data = AdaptationTraining()
    training_loader = DataLoader(training_data, batch_size=8, shuffle=True, num_workers=0,
                                generator=torch.Generator().manual_seed(SEED))
    # Start this run from the original checkpoint, not a previous fine-tuned model.
    adapted_model = Model(config['model_config']).to(device)
    adapted_model.load_state_dict(torch.load(checkpoint, map_location=device, weights_only=True), strict=True)
    adapted_model.train()
    optimizer = torch.optim.Adam(adapted_model.parameters(), lr=1e-6)
    criterion = torch.nn.CrossEntropyLoss(weight=torch.tensor([0.1, 0.9], device=device))
    loss_sum, seen = 0.0, 0
    if device.type == 'cuda':
        torch.cuda.synchronize()
    started = time.perf_counter()
    print('Training started. Output folder:', run_dir)
    for batch_number, (x, y) in enumerate(training_loader, start=1):
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad(set_to_none=True)
        _, logits = adapted_model(x)
        loss = criterion(logits, y)
        assert torch.isfinite(loss), 'Nonfinite training loss; stop and investigate.'
        loss.backward()
        for parameter in adapted_model.parameters():
            if parameter.grad is not None:
                assert torch.isfinite(parameter.grad).all(), 'Nonfinite gradient'
        optimizer.step()
        seen += len(y)
        loss_sum += loss.item() * len(y)
        if batch_number % 50 == 0:
            print(f'Batch {batch_number}/{len(training_loader)} | recordings {seen}/4000 | loss {loss.item():.4f}')

    assert seen == 4000
    if device.type == 'cuda':
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - started
    adapted_model.eval()
    # Atomic replacement prevents a partial file being mistaken for a finished checkpoint.
    temporary = run_dir / 'adapted_epoch1.pth.partial'
    torch.save(adapted_model.state_dict(), temporary)
    os.replace(temporary, run_dir / 'adapted_epoch1.pth')
    training_log = {'epoch': 1, 'recordings': seen, 'filtered_recordings': training_data.filtered_count,
                    'batch_loss_mean_weighted_by_batch_size': loss_sum / seen, 'training_seconds': elapsed}
    (run_dir / 'training_log.json').write_text(json.dumps(training_log, indent=2))
    print('TRAINING COMPLETE:', training_log)
    print('Checkpoint:', run_dir / 'adapted_epoch1.pth')
    clean_control_dir = run_dir
    print('CLEAN CONTROL COMPLETE:', clean_control_dir)
    print('Original model unchanged. Run step 13 for development evaluation.')


Using device: cuda
Training started. Output folder: /content/runs/run_20261118T064542491901Z/clean_control_20261118T064915641274Z


KeyboardInterrupt: 

#### C14. Evaluate the clean-only control
Evaluate the clean-only checkpoint using the same development conditions.


In [33]:
if not RUN_EXPERIMENTS:
    print("Skipped: optional experiment. No model or result files changed.")
else:
    import csv, json, hashlib
    from pathlib import Path
    import numpy as np
    import soundfile as sf
    import torch
    from torch.utils.data import Dataset, DataLoader
    from scipy.signal import butter, sosfilt
    from models.AASIST import Model
    from data_utils import genSpoof_list, pad
    from evaluation import compute_eer

    # Evaluate this completed trial explicitly; do not rerun training.
    assert 'clean_control_dir' in globals(), 'Run step 12 first, or set clean_control_dir to the saved control folder.'
    trial_dir = Path(clean_control_dir)
    adapted_checkpoint = trial_dir / 'adapted_epoch1.pth'
    assert adapted_checkpoint.is_file(), f'Missing checkpoint: {adapted_checkpoint}'
    print('Using device:', device)
    settings = json.loads((trial_dir / 'settings.json').read_text())
    protocol = DATA / 'ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.dev.trl.txt'
    eval_dev_labels, eval_dev_ids = genSpoof_list(str(protocol), is_train=False, is_eval=False)
    assert len(eval_dev_ids) == len(set(eval_dev_ids)) == 24844

    # Bind cached scores to this exact checkpoint and evaluation policy.
    provenance = {
        'checkpoint_sha256': hashlib.sha256(adapted_checkpoint.read_bytes()).hexdigest(),
        'protocol_sha256': hashlib.sha256(protocol.read_bytes()).hexdigest(),
        'crop': 'first 64600 samples; repeat short audio',
        'filter': 'butter N=6, 300-3400 Hz, SOS, sosfilt before crop',
        'score': 'class-1 raw bona-fide logit'
    }
    metadata_file = trial_dir / 'development_scoring_settings.json'
    if metadata_file.exists():
        assert json.loads(metadata_file.read_text()) == provenance, 'Cached experiment settings differ'
    else:
        assert not list(trial_dir.glob('adapted_dev_*_scores.csv')), 'Existing scores lack provenance'
        metadata_file.write_text(json.dumps(provenance, indent=2))

    # Reload the saved adapted weights; leave the baseline model untouched.
    evaluation_model = Model(settings['model_config']).to(device)
    evaluation_model.load_state_dict(torch.load(adapted_checkpoint, map_location=device, weights_only=True), strict=True)
    evaluation_model.eval()

    class AdaptedDevelopment(Dataset):
        def __init__(self, filtered):
            self.filtered = filtered
            self.sos = butter(6, [300, 3400], btype='bandpass', fs=16000, output='sos')
        def __len__(self):
            return len(eval_dev_ids)
        def __getitem__(self, i):
            uid = eval_dev_ids[i]
            y, sr = sf.read(DATA / 'ASVspoof2019_LA_dev/flac' / f'{uid}.flac')
            assert sr == 16000 and y.ndim == 1 and y.size > 0 and np.isfinite(y).all()
            if self.filtered:
                y = sosfilt(self.sos, y)
            x = torch.from_numpy(np.asarray(pad(y, max_len=64600), dtype=np.float32))
            assert x.shape == (64600,) and torch.isfinite(x).all()
            return x, uid

    def adapted_scores(condition, filtered):
        path = trial_dir / f'adapted_dev_{condition}_scores.csv'
        if path.exists():
            with path.open(newline='') as stream:
                records = list(csv.DictReader(stream))
            print('Reusing completed:', path.name)
        else:
            records = []
            batches = DataLoader(AdaptedDevelopment(filtered), batch_size=8, shuffle=False, num_workers=0)
            with torch.inference_mode():
                for i, (x, uids) in enumerate(batches, start=1):
                    _, logits = evaluation_model(x.to(device))
                    for uid, score in zip(uids, logits[:, 1].cpu().tolist()):
                        records.append({'recording_id': uid, 'true_label': int(eval_dev_labels[uid]),
                                        'bona_fide_score': float(score)})
                    if i % 100 == 0:
                        print(f'{condition}: {len(records)}/24844')
        assert [r['recording_id'] for r in records] == eval_dev_ids
        assert all(int(r['true_label']) == eval_dev_labels[r['recording_id']] for r in records)
        scores = np.array([float(r['bona_fide_score']) for r in records])
        assert np.isfinite(scores).all()
        if not path.exists():
            temporary = path.with_suffix('.csv.partial')
            with temporary.open('w', newline='') as stream:
                writer = csv.DictWriter(stream, fieldnames=['recording_id', 'true_label', 'bona_fide_score'])
                writer.writeheader()
                writer.writerows(records)
            temporary.replace(path)
        print('Completed:', condition)
        return scores

    adapted_clean_scores = adapted_scores('clean', False)
    adapted_filtered_scores = adapted_scores('bandlimited', True)
    y = np.array([eval_dev_labels[uid] for uid in eval_dev_ids])
    genuine, spoof = y == 1, y == 0
    _, adapted_threshold = compute_eer(adapted_clean_scores[genuine], adapted_clean_scores[spoof])
    adapted_metrics = []
    for condition, scores in [('clean', adapted_clean_scores), ('bandlimited', adapted_filtered_scores)]:
        eer, _ = compute_eer(scores[genuine], scores[spoof])
        predicted_genuine = scores > adapted_threshold
        flagged = int(np.sum(genuine & ~predicted_genuine))
        missed = int(np.sum(spoof & predicted_genuine))
        record = {'condition': condition, 'eer': float(eer),
                  'clean_development_threshold': float(adapted_threshold),
                  'genuine_total': int(genuine.sum()), 'genuine_flagged': flagged,
                  'genuine_false_alarm_rate': flagged / int(genuine.sum()),
                  'spoof_total': int(spoof.sum()), 'spoof_missed': missed,
                  'spoof_miss_rate': missed / int(spoof.sum())}
        adapted_metrics.append(record)
        print(f'CLEAN-CONTROL {condition}: EER {eer*100:.3f}% | genuine flagged {flagged}/{genuine.sum()} | spoof accepted {missed}/{spoof.sum()}')
    metrics_path = trial_dir / 'adapted_dev_metrics.csv'
    with metrics_path.open('w', newline='') as stream:
        writer = csv.DictWriter(stream, fieldnames=list(adapted_metrics[0]))
        writer.writeheader()
        writer.writerows(adapted_metrics)

    baseline_path = trial_dir.parent / 'aasist_original_dev_full_metrics.csv'
    if baseline_path.exists():
        with baseline_path.open(newline='') as stream:
            baseline = {r['condition']: r for r in csv.DictReader(stream)}
        print('Comparison (negative EER change means improvement):')
        for r in adapted_metrics:
            original_eer = float(baseline[r['condition']]['eer'])
            print(f"{r['condition']}: original {original_eer*100:.3f}% -> adapted {r['eer']*100:.3f}% | change {(r['eer']-original_eer)*100:+.3f} percentage points")
    print('Adapted clean development threshold:', adapted_threshold)
    print('Metrics saved:', metrics_path)
    print('Development evaluation complete. Final evaluation partition remains untouched.')

    # Read the completed augmented trial for a three-way development comparison.
    augmented_metrics_path = Path(augmented_trial_dir) / 'adapted_dev_metrics.csv'
    if augmented_metrics_path.exists() and baseline_path.exists():
        with augmented_metrics_path.open(newline='') as stream:
            augmented_metrics = {r['condition']: r for r in csv.DictReader(stream)}
        print('THREE-WAY DEVELOPMENT EER (%)')
        for r in adapted_metrics:
            condition = r['condition']
            print(condition, '| original:', round(float(baseline[condition]['eer'])*100, 3),
                  '| clean-only:', round(r['eer']*100, 3),
                  '| augmented:', round(float(augmented_metrics[condition]['eer'])*100, 3))


AssertionError: Run step 12 first, or set clean_control_dir to the saved control folder.

#### C15. Freeze settings and thresholds
Record the checkpoint hashes and development thresholds before held-out evaluation.


In [ ]:
if not RUN_EXPERIMENTS:
    print("Skipped: optional experiment. No model or result files changed.")
else:
    import csv, json, hashlib, subprocess
    from pathlib import Path
    from datetime import datetime, timezone
    import torch, numpy as np

    ROOT = Path(f'{ROOT}')
    original_checkpoint = Path(config['model_path']).resolve()
    model_sources = [
        ('original', original_checkpoint, ROOT / 'aasist_original_dev_full_metrics.csv', config['model_config']),
        ('clean_only', Path(clean_control_dir) / 'adapted_epoch1.pth',
         Path(clean_control_dir) / 'adapted_dev_metrics.csv', None),
        ('augmented', Path(augmented_trial_dir) / 'adapted_epoch1.pth',
         Path(augmented_trial_dir) / 'adapted_dev_metrics.csv', None)
    ]
    final_models = []
    for name, weights_path, metrics_path, architecture in model_sources:
        assert weights_path.is_file() and metrics_path.is_file(), f'Missing files for {name}'
        if architecture is None:
            architecture = json.loads((weights_path.parent / 'settings.json').read_text())['model_config']
        with metrics_path.open(newline='') as stream:
            clean_records = [r for r in csv.DictReader(stream) if r['condition'] == 'clean']
        assert len(clean_records) == 1
        threshold = float(clean_records[0]['clean_development_threshold'])
        assert np.isfinite(threshold)
        final_models.append({'name': name, 'checkpoint': str(weights_path),
            'checkpoint_sha256': hashlib.sha256(weights_path.read_bytes()).hexdigest(),
            'threshold': threshold, 'threshold_source': str(metrics_path), 'model_config': architecture})
    assert final_models[0]['checkpoint_sha256'] == '51d2d9cf0738172f61e2a384ec50a54a55363240f67c971ed55a92435bc1a1c0', 'Original checkpoint differs'

    FINAL_DIR = ROOT / ('final_evaluation_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'))
    FINAL_DIR.mkdir(exist_ok=False)
    plan = {'models': final_models, 'data': 'ASVspoof2019 LA official evaluation partition',
        'conditions': ['clean', 'bandlimited'], 'filter': 'butter N=6 300-3400 Hz SOS sosfilt before crop',
        'input': '16 kHz mono, first 64600 samples or repeat short recordings',
        'batch_size': 8, 'decision': 'score > model clean-development threshold means genuine',
        'torch': torch.__version__, 'numpy': np.__version__,
        'source_commit': subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()}
    (FINAL_DIR / 'frozen_plan.json').write_text(json.dumps(plan, indent=2))
    print('Frozen plan saved:', FINAL_DIR / 'frozen_plan.json')
    for entry in final_models:
        print(entry['name'], 'development threshold:', entry['threshold'])
    print('Next: step 15. No final evaluation scores have been calculated in this cell.')


#### C16. Run held-out final evaluation
Score each frozen model on all 71,237 evaluation recordings in both conditions. Save completed conditions for reuse; an interrupted condition must restart.


In [ ]:
if not RUN_EXPERIMENTS:
    print("Skipped: optional experiment. No model or result files changed.")
else:
    import csv, json, hashlib, gc
    from pathlib import Path
    import numpy as np
    import soundfile as sf
    import torch
    from torch.utils.data import Dataset, DataLoader
    from scipy.signal import butter, sosfilt
    from models.AASIST import Model
    from data_utils import genSpoof_list, pad
    from evaluation import compute_eer

    # If reconnecting, restore steps 1-4, then set FINAL_DIR to the existing frozen-plan folder.
    assert 'FINAL_DIR' in globals(), 'Run step 14 first, or restore the saved FINAL_DIR path.'
    print('Using device:', device)
    plan = json.loads((FINAL_DIR / 'frozen_plan.json').read_text())
    protocol = DATA / 'ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.eval.trl.txt'
    final_labels, final_ids = genSpoof_list(str(protocol), is_train=False, is_eval=False)
    assert len(final_ids) == len(set(final_ids)) == 71237
    for split, suffix in [('train', 'trn'), ('dev', 'trl')]:
        other_protocol = DATA / 'ASVspoof2019_LA_cm_protocols' / f'ASVspoof2019.LA.cm.{split}.{suffix}.txt'
        _, other_ids = genSpoof_list(str(other_protocol), is_train=False, is_eval=False)
        assert not set(final_ids).intersection(other_ids), 'Partition recording-ID overlap'
    missing = [uid for uid in final_ids if not (DATA / 'ASVspoof2019_LA_eval/flac' / f'{uid}.flac').is_file()]
    assert not missing, f'Missing evaluation files (first 10): {missing[:10]}'
    y = np.array([final_labels[uid] for uid in final_ids])
    genuine, spoof = y == 1, y == 0
    assert int(genuine.sum()) == 7355 and int(spoof.sum()) == 63882
    print('Held-out evaluation recordings:', len(final_ids))

    class FinalAudio(Dataset):
        def __init__(self, filtered):
            self.filtered = filtered
            self.sos = butter(6, [300, 3400], btype='bandpass', fs=16000, output='sos')
        def __len__(self):
            return len(final_ids)
        def __getitem__(self, i):
            uid = final_ids[i]
            audio, sr = sf.read(DATA / 'ASVspoof2019_LA_eval/flac' / f'{uid}.flac')
            assert sr == 16000 and audio.ndim == 1 and audio.size and np.isfinite(audio).all()
            if self.filtered:
                audio = sosfilt(self.sos, audio)
            x = torch.from_numpy(np.asarray(pad(audio, max_len=64600), dtype=np.float32))
            assert torch.isfinite(x).all()
            return x, uid

    summary = []
    for entry in plan['models']:
        weights_path = Path(entry['checkpoint'])
        assert hashlib.sha256(weights_path.read_bytes()).hexdigest() == entry['checkpoint_sha256']
        scoring_model = Model(entry['model_config']).to(device)
        scoring_model.load_state_dict(torch.load(weights_path, map_location=device, weights_only=True), strict=True)
        scoring_model.eval()
        for condition in plan['conditions']:
            path = FINAL_DIR / f"{entry['name']}_{condition}_scores.csv"
            if path.exists():
                with path.open(newline='') as stream:
                    records = list(csv.DictReader(stream))
                print('Reusing completed scores:', path.name)
            else:
                records = []
                loader = DataLoader(FinalAudio(condition == 'bandlimited'), batch_size=8, shuffle=False, num_workers=0)
                with torch.inference_mode():
                    for i, (x, uids) in enumerate(loader, start=1):
                        _, logits = scoring_model(x.to(device))
                        for uid, score in zip(uids, logits[:, 1].cpu().tolist()):
                            records.append({'recording_id': uid, 'true_label': final_labels[uid], 'bona_fide_score': float(score)})
                        if i % 100 == 0:
                            print(entry['name'], condition, f'{len(records)}/{len(final_ids)}')
            assert [r['recording_id'] for r in records] == final_ids
            assert all(int(r['true_label']) == final_labels[r['recording_id']] for r in records)
            scores = np.array([float(r['bona_fide_score']) for r in records])
            assert np.isfinite(scores).all()
            if not path.exists():
                temp = path.with_suffix('.csv.partial')
                with temp.open('w', newline='') as stream:
                    writer = csv.DictWriter(stream, fieldnames=['recording_id', 'true_label', 'bona_fide_score'])
                    writer.writeheader(); writer.writerows(records)
                temp.replace(path)
            eer, _ = compute_eer(scores[genuine], scores[spoof])
            # Evaluation EER is descriptive only: its threshold is discarded.
            prediction = scores > entry['threshold']
            flagged = int(np.sum(genuine & ~prediction))
            missed = int(np.sum(spoof & prediction))
            summary.append({'model': entry['name'], 'condition': condition, 'eer': float(eer),
                'development_threshold': entry['threshold'], 'genuine_total': int(genuine.sum()),
                'genuine_flagged': flagged, 'genuine_false_alarm_rate': flagged / int(genuine.sum()),
                'spoof_total': int(spoof.sum()), 'spoof_missed': missed, 'spoof_miss_rate': missed / int(spoof.sum())})
            # Persist metrics after every completed condition.
            with (FINAL_DIR / 'final_metrics.csv').open('w', newline='') as stream:
                writer = csv.DictWriter(stream, fieldnames=list(summary[0]))
                writer.writeheader(); writer.writerows(summary)
            print(entry['name'], condition, f'EER {eer*100:.3f}%', 'genuine flagged:', flagged, 'spoof missed:', missed)
        del scoring_model
        gc.collect()
        if device.type == 'cuda':
            torch.cuda.empty_cache()
    print('FINAL EVALUATION COMPLETE:', FINAL_DIR / 'final_metrics.csv')
    print('Do not tune settings based on these held-out results.')


#### C17. Package the new results
Package the results of a completed reproduction run. This excludes the dataset.


In [ ]:
if not RUN_EXPERIMENTS:
    print('Skipped: no experiment results packaged.')
else:
    assert (FINAL_DIR / 'final_metrics.csv').is_file(), 'Complete C16 first.'
    import csv
    with (FINAL_DIR / 'final_metrics.csv').open(newline='') as stream:
        assert len(list(csv.DictReader(stream))) == 6, 'Expected six final metric rows'
    result_zip = shutil.make_archive(str(BASE / ROOT.name), 'zip', root_dir=ROOT.parent, base_dir=ROOT.name)
    print('RESULT ZIP:', result_zip)
    print('Colab: download this ZIP from the Files panel before disconnecting.')


### Sources and assistance

- [AASIST source, pretrained weights and MIT licence](https://github.com/clovaai/aasist) .
- [AASIST paper](https://arxiv.org/abs/2110.01200).
- [ASVspoof 2019 dataset](https://zenodo.org/records/6906306).
- [Supporting communication-robustness paper](https://arxiv.org/abs/2504.12423). It studies codecs and packet loss; this experiment uses bandwidth limitation only.
- [SciPy filter design](https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.butter.html) and [filter application](https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.sosfilt.html).
